# F2 · Obtención, exploración, limpieza, transformación y validación
**Proyecto:** Análisis reproducible del Inventario Público de Glaciares de Chile 2022  
**Grupo 3:** Aníbal Reyes · Francisco Correa Soto · Gustavo Aloy · Roberto Gaubert  
**Asignatura:** MCDI500 · Programación para la Ciencia de Datos

**Pregunta:** ¿Cómo se distribuyen los glaciares del Inventario Público de Glaciares 2022 según región, clasificación, superficie y volumen estimado?

**Alcance:** este notebook prepara y valida el inventario; los resúmenes territoriales finales y las visualizaciones analíticas corresponden a F3–F4. Las estimaciones de volumen no equivalen a mediciones directas. Cada decisión se fundamenta en evidencia obtenida del CSV. No se altera el archivo original.

## 0. Entorno reproducible y rutas
Se verifican las dependencias efectivamente utilizadas. El notebook puede ejecutarse desde `F2/` o desde la raíz del repositorio. Se usan rutas relativas al proyecto y no se necesita acceso a internet.

In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd

RAIZ = Path.cwd().resolve()
if not (RAIZ / "data" / "raw").exists():
    RAIZ = RAIZ.parent
ENTRADA = RAIZ / "data" / "raw" / "Inventario_Publico_de_Glaciares_2022.csv"
SALIDA = RAIZ / "data" / "processed"
assert ENTRADA.is_file(), f"No se encontró el inventario: {ENTRADA}"
print("Python:", sys.version.split()[0], "| NumPy:", np.__version__, "| Pandas:", pd.__version__)
print("Raíz:", RAIZ)
print("Fuente:", ENTRADA.relative_to(RAIZ))


Python: 3.13.5 | NumPy: 2.3.5 | Pandas: 2.2.3
Raíz: /mnt/data/Glaciares_Grupo3
Fuente: data/raw/Inventario_Publico_de_Glaciares_2022.csv


## 1. Obtención y preservación de la fuente
El inventario se carga sin alterar sus 45 columnas. Los códigos territoriales se preservan como aparecen en el archivo; no se infiere que representen cantidades. Se crea una copia independiente para el procesamiento. Se registran dimensiones y una huella SHA-256 del archivo para trazabilidad.

In [2]:
import hashlib

def huella_sha256(ruta):
    h = hashlib.sha256()
    with open(ruta, "rb") as archivo:
        for bloque in iter(lambda: archivo.read(1024 * 1024), b""):
            h.update(bloque)
    return h.hexdigest()

def cargar_inventario(ruta):
    return pd.read_csv(ruta, low_memory=False)

huella_inicial = huella_sha256(ENTRADA)
original = cargar_inventario(ENTRADA)
datos = original.copy(deep=True)
print("Dimensiones originales:", original.shape)
print("SHA-256:", huella_inicial)
display(original.head(3))


Dimensiones originales: (26180, 45)
SHA-256: 5691de5d7d7ffe2c8586a7066e227ac5cc91fa1ec69f54096e937a054c283f1d


,OBJECTID,Id,COD_GLA,NOMBRE,CLASIFICA,AREA_KM2,REGION,PROVINCIA,COMUNA,DATUM,HUSO,NORTE,ESTE,FUENTE_DIG,FUEN_FECHA,INVE_FECHA,NOM_CUEN,COD_CUEN,COD_SCUEN,COD_SSCUEN,MZON_GLAC,CH_CASQ,CUBIERTO,LATITUD,LONGITUD,PERIMETRO,ORIENTA,HMEDIA,HMAX,HMIN,PENDIENTE,ESP_MED,F_ESP_MED,VOL_km3,EQ_AGUAKM3,WGI_1,WGI_2,WGI_3,FRENTE_TER,RESOL_IMG,ERROR,OBSERVACIO,CUT_REG,CUT_PROV,CUT_COM
0,174,0,CL101010001@,VOLCAN PARINACOTA,GLACIAR DE MONTAÑA,1.168,ARICA Y PARINACOTA,PARINACOTA,PUTRE,WGS 84,19 SUR,7990505,484834,LANDSAT OLI,05/12/2015,2019,ALTIPLANICAS,10,101,1010,MACROZONA NORTE,NaN,NaN,"-18,1744803305003","-69,1433999344572",7.384,S,5804.476,6317.0,5375.0,30.759,30.121,CHEN & OHMURA 1990,0.03517,0.02989,6,6,3,TIERRA,15.0,0.11076,NaN,15,152,15201
1,180,0,CL101001023@,VOLCAN POMERAPE,GLACIAR DE MONTAÑA,0.478,ARICA Y PARINACOTA,PARINACOTA,PUTRE,WGS 84,19 SUR,7996034,486336,LANDSAT OLI,05/12/2015,2019,ALTIPLANICAS,10,100,1001,MACROZONA NORTE,NaN,NaN,"-18,1245109467846","-69,1291611785678",5.916,NW,6071.467,6255.0,5664.0,28.494,21.890,CHEN & OHMURA 1990,0.01045,0.00889,6,9,6,TIERRA,15.0,0.08874,NaN,15,152,15201
2,183,0,CL101021004@,S/N,GLACIARETE,0.162,ARICA Y PARINACOTA,PARINACOTA,PUTRE,WGS 84,19 SUR,7962456,489894,LANDSAT OLI,05/12/2015,2019,ALTIPLANICAS,10,102,1021,MACROZONA NORTE,NaN,NaN,"-18,4280225838322","-69,0956929210804",3.498,S,5840.866,5947.0,5685.0,27.344,14.887,CHEN & OHMURA 1990,0.00241,0.00205,7,7,3,TIERRA,15.0,0.05248,NaN,15,152,15201


## 2. Exploración y diagnóstico previo
Antes de modificar datos, se cuantifican tipos, faltantes, unicidad y duplicados. Se examinan rangos de las magnitudes relevantes. Los valores extremos no se eliminan automáticamente: los glaciares de gran superficie pueden ser observaciones reales.

In [3]:
def perfil_columnas(df):
    return pd.DataFrame({
        "variable": df.columns,
        "tipo_original": [str(df[c].dtype) for c in df.columns],
        "no_nulos": [int(df[c].notna().sum()) for c in df.columns],
        "faltantes": [int(df[c].isna().sum()) for c in df.columns],
        "pct_faltantes": [round(df[c].isna().mean() * 100, 2) for c in df.columns],
        "valores_unicos": [int(df[c].nunique(dropna=True)) for c in df.columns],
    })

perfil_inicial = perfil_columnas(original)
display(perfil_inicial)
print("Filas duplicadas exactas:", int(original.duplicated().sum()))
print("COD_GLA duplicados:", int(original.COD_GLA.duplicated().sum()))
print("Códigos faltantes:", int(original.COD_GLA.isna().sum()))


,variable,tipo_original,no_nulos,faltantes,pct_faltantes,valores_unicos
0,OBJECTID,int64,26180,0,0.00,26078
1,Id,int64,26180,0,0.00,1
2,COD_GLA,object,26180,0,0.00,26180
3,NOMBRE,object,26180,0,0.00,431
4,CLASIFICA,object,26180,0,0.00,5
5,AREA_KM2,float64,26180,0,0.00,2323
6,REGION,object,26180,0,0.00,16
7,PROVINCIA,object,26180,0,0.00,40
8,COMUNA,object,26180,0,0.00,84
9,DATUM,object,26180,0,0.00,1


Filas duplicadas exactas: 0
COD_GLA duplicados: 0
Códigos faltantes: 0


In [4]:
VARIABLES_FISICAS = ["AREA_KM2", "VOL_km3", "EQ_AGUAKM3", "ESP_MED", "HMIN", "HMEDIA", "HMAX", "PENDIENTE"]
resumen_fisico = original[VARIABLES_FISICAS].describe().T
resumen_fisico["ceros"] = (original[VARIABLES_FISICAS] == 0).sum()
display(resumen_fisico)

def diagnostico_iqr(df, columnas):
    filas = []
    for columna in columnas:
        x = pd.to_numeric(df[columna], errors="coerce").dropna()
        q1, q3 = x.quantile([.25, .75])
        iqr = q3 - q1
        fuera = int(((x < q1 - 1.5 * iqr) | (x > q3 + 1.5 * iqr)).sum())
        filas.append({"variable": columna, "q1": q1, "q3": q3, "fuera_iqr": fuera})
    return pd.DataFrame(filas)

display(diagnostico_iqr(original, ["AREA_KM2", "VOL_km3", "EQ_AGUAKM3"]))
print("Fechas no interpretables con formato DD/MM/AAAA:",
      int(pd.to_datetime(original.FUEN_FECHA, format="%d/%m/%Y", errors="coerce").isna().sum()))


,count,mean,std,min,25%,50%,75%,max,ceros
AREA_KM2,26180.0,0.802610,13.003948,0.001,0.01100,0.03600,0.152000,772.51100,0
VOL_km3,26180.0,0.115592,3.506222,0.000,0.00006,0.00028,0.001990,236.44572,362
EQ_AGUAKM3,26180.0,0.098268,2.980288,0.000,0.00005,0.00025,0.001713,200.97886,668
ESP_MED,26180.0,12.363116,13.135727,2.127,5.74175,8.73650,14.546500,306.07400,0
HMIN,26180.0,1815.409740,1261.124660,0.000,1011.00000,1421.00000,1810.000000,6726.00000,69
HMEDIA,26180.0,1935.049220,1245.989177,27.940,1126.05650,1524.61300,1899.180250,6734.63100,0
HMAX,26180.0,2047.579030,1262.929441,111.000,1228.00000,1616.00000,2073.250000,6764.00000,0
PENDIENTE,26180.0,25.461596,7.703104,1.119,20.47600,25.45600,30.256250,74.30000,0


,variable,q1,q3,fuera_iqr
0,AREA_KM2,0.01100,0.152000,3761
1,VOL_km3,0.00006,0.001990,4380
2,EQ_AGUAKM3,0.00005,0.001713,4377


Fechas no interpretables con formato DD/MM/AAAA: 13


## 3. Decisiones de limpieza justificadas

| Evidencia | Alternativas | Decisión para F2 |
|---|---|---|
| `CUBIERTO` y `OBSERVACIO` sin información | Eliminar, imputar, conservar | **Conservar** las columnas originales sin imputar; excluirlas solo de futuros análisis por carecer de información. |
| `CH_CASQ` mayoritariamente vacío | Eliminar, completar, conservar | **Conservar** los faltantes: no hay una imputación respaldada por la fuente. |
| `LATITUD` y `LONGITUD` como texto con coma decimal | Conservar texto o convertir | Crear columnas numéricas auxiliares sin sobrescribir los valores originales. |
| `FUEN_FECHA` como texto | Mantener texto o convertir | Crear columna de fecha auxiliar; valores no interpretables quedan como `NaT` y se cuantifican. |
| Variables físicas con distribución asimétrica | Imputar media, mediana, por grupo, eliminar | No imputar sin necesidad: las variables principales no presentan faltantes; no eliminar extremos sin evidencia de error. |
| `Id` constante | Eliminar o conservar | Conservar como parte del registro fuente, sin utilizarlo para identificar glaciares. |

**Nota:** la normalización, el escalamiento y la codificación one-hot no son necesarios para preparar los resúmenes descriptivos de F3; aplicar estas técnicas ahora cambiaría la interpretación física o generaría columnas redundantes. Los códigos WGI y territoriales se mantienen como códigos. Se evita confundir limpieza con eliminación indiscriminada.

## 4. Transformaciones reproducibles
Se encapsulan en funciones independientes y no se modifica el DataFrame original. Las columnas auxiliares hacen explícita la interpretación de coordenadas y fechas. Si se encuentran valores inesperados, se conservan como faltantes en las columnas auxiliares y se reportan para revisión; no se inventan valores.

In [5]:
def convertir_decimal_coma(serie):
    return pd.to_numeric(serie.astype("string").str.strip().str.replace(",", ".", regex=False), errors="coerce")

def transformar_inventario(df):
    resultado = df.copy(deep=True)
    resultado["LATITUD_NUM"] = convertir_decimal_coma(resultado["LATITUD"])
    resultado["LONGITUD_NUM"] = convertir_decimal_coma(resultado["LONGITUD"])
    resultado["FUEN_FECHA_DT"] = pd.to_datetime(resultado["FUEN_FECHA"], format="%d/%m/%Y", errors="coerce")
    return resultado

datos_procesados = transformar_inventario(datos)
print("Dimensiones después:", datos_procesados.shape)
print("Nuevas columnas:", [c for c in datos_procesados if c not in original])


Dimensiones después: (26180, 48)
Nuevas columnas: ['LATITUD_NUM', 'LONGITUD_NUM', 'FUEN_FECHA_DT']


## 5. Impacto cuantificado de las decisiones
Se compara la disponibilidad de valores antes y después. No se imputan superficies ni volúmenes, de modo que sus estadísticas deben permanecer idénticas. Se registran conversiones fallidas y cualquier anomalía de coherencia física sin corregirla automáticamente.

In [6]:
def reporte_impacto(antes, despues):
    return pd.DataFrame([
        {"control": "Registros", "antes": len(antes), "despues": len(despues)},
        {"control": "Columnas", "antes": antes.shape[1], "despues": despues.shape[1]},
        {"control": "Latitudes numéricas", "antes": int(pd.api.types.is_numeric_dtype(antes.LATITUD)), "despues": int(despues.LATITUD_NUM.notna().sum())},
        {"control": "Longitudes numéricas", "antes": int(pd.api.types.is_numeric_dtype(antes.LONGITUD)), "despues": int(despues.LONGITUD_NUM.notna().sum())},
        {"control": "Fechas interpretables", "antes": 0, "despues": int(despues.FUEN_FECHA_DT.notna().sum())},
        {"control": "Nulos en AREA_KM2", "antes": int(antes.AREA_KM2.isna().sum()), "despues": int(despues.AREA_KM2.isna().sum())},
        {"control": "Nulos en VOL_km3", "antes": int(antes.VOL_km3.isna().sum()), "despues": int(despues.VOL_km3.isna().sum())},
    ])

display(reporte_impacto(original, datos_procesados))
print("Latitudes fallidas:", int(datos_procesados.LATITUD_NUM.isna().sum()))
print("Longitudes fallidas:", int(datos_procesados.LONGITUD_NUM.isna().sum()))
print("Fechas no interpretables:", int(datos_procesados.FUEN_FECHA_DT.isna().sum()))
print("Orden altitudinal anómalo (HMIN <= HMEDIA <= HMAX):",
      int((~((datos_procesados.HMIN <= datos_procesados.HMEDIA) & (datos_procesados.HMEDIA <= datos_procesados.HMAX))).sum()))


,control,antes,despues
0,Registros,26180,26180
1,Columnas,45,48
2,Latitudes numéricas,0,26180
3,Longitudes numéricas,0,26180
4,Fechas interpretables,0,26167
5,Nulos en AREA_KM2,0,0
6,Nulos en VOL_km3,0,0


Latitudes fallidas: 0
Longitudes fallidas: 0
Fechas no interpretables: 13
Orden altitudinal anómalo (HMIN <= HMEDIA <= HMAX): 1


## 6. Validación técnica y casos límite
Las comprobaciones automáticas distinguen invariantes obligatorias de advertencias que requieren revisión. No se obliga a que todos los registros sean perfectos; se verifica que los problemas permanezcan visibles y que el pipeline no pierda información sin justificación.

In [7]:
def validar_inventario(antes, despues):
    assert len(antes) == len(despues), "Se perdieron registros"
    assert list(despues.columns[:antes.shape[1]]) == list(antes.columns), "Se alteró el esquema original"
    pd.testing.assert_frame_equal(antes, despues[antes.columns], check_dtype=True)
    assert antes.COD_GLA.notna().all(), "Identificadores faltantes"
    assert antes.COD_GLA.is_unique, "Identificadores duplicados"
    assert despues.LATITUD_NUM.between(-90, 90).all(), "Latitud fuera de rango o nula"
    assert despues.LONGITUD_NUM.between(-180, 180).all(), "Longitud fuera de rango o nula"
    for columna in ["AREA_KM2", "VOL_km3", "EQ_AGUAKM3"]:
        assert despues[columna].notna().all(), f"Faltantes en {columna}"
        assert (despues[columna] >= 0).all(), f"Valores negativos en {columna}"
    return True

assert validar_inventario(original, datos_procesados)
assert pd.isna(convertir_decimal_coma(pd.Series(["dato inválido"])).iloc[0])
assert convertir_decimal_coma(pd.Series(["-33,45"])).iloc[0] == -33.45
assert transformar_inventario(original.head(0)).empty
assert transformar_inventario(original.head(1)).shape == (1, original.shape[1] + 3)
assert huella_sha256(ENTRADA) == huella_inicial
print("Pruebas superadas: preservación, identificadores, rangos, no negatividad, conversión, caso vacío y huella de origen.")


Pruebas superadas: preservación, identificadores, rangos, no negatividad, conversión, caso vacío y huella de origen.


## 7. Exportación del conjunto procesado y evidencia
Se conserva el CSV original en `data/raw/`. El resultado se escribe en `data/processed/` como CSV, con las 45 columnas originales más tres columnas auxiliares. Las fechas se serializan en formato ISO y se acompañan de un perfil de salida. Los resultados descriptivos finales se reservan para F3–F4.

In [8]:
SALIDA.mkdir(parents=True, exist_ok=True)
ruta_procesado = SALIDA / "inventario_glaciares_2022_procesado.csv"
ruta_perfil = SALIDA / "perfil_f2.csv"
datos_procesados.to_csv(ruta_procesado, index=False, date_format="%Y-%m-%d")
perfil_final = perfil_columnas(datos_procesados)
perfil_final.to_csv(ruta_perfil, index=False)
relectura = pd.read_csv(ruta_procesado, low_memory=False)
assert relectura.shape == datos_procesados.shape
assert relectura.COD_GLA.equals(datos_procesados.COD_GLA)
assert huella_sha256(ENTRADA) == huella_inicial
print("Datos procesados:", ruta_procesado.relative_to(RAIZ))
print("Perfil final:", ruta_perfil.relative_to(RAIZ))
print("Dimensiones exportadas:", relectura.shape)


Datos procesados: data/processed/inventario_glaciares_2022_procesado.csv
Perfil final: data/processed/perfil_f2.csv
Dimensiones exportadas: (26180, 48)


## 8. Bitácora de decisiones y continuidad
**F2 implementado:** carga sin modificación de la fuente, diagnóstico de las 45 columnas, decisión de no imputar ni eliminar datos sin justificación, conversión auxiliar de coordenadas y fecha, validación y exportación reproducible.

**Pendiente de verificación documental:** significado y unidades exactas de `ERROR`, `PERIMETRO`, `PENDIENTE`, `CH_CASQ` y códigos WGI. Las 13 fechas no interpretables y la anomalía altitudinal se reportan, pero no se corrigen sin evidencia externa.

**F3:** generar agregaciones por región, macrozona, cuenca y clasificación; **F4:** visualizar y comunicar los resultados. No sumar volúmenes como si fueran mediciones directas ni interpretar una foto transversal como tendencia temporal.

**Trazabilidad Git:** registrar en commits separados el diagnóstico, las transformaciones, las validaciones y los resultados exportados; completar referencias a hashes de commits del equipo en el informe. **Prueba externa pendiente:** otro integrante debe clonar, instalar dependencias y ejecutar `Restart & Run All` para demostrar portabilidad.